In [1]:

import pandas as pd
import torch
from pandas import DataFrame
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM

# Data preparation

In [2]:
# Model parameters
HORIZON = 1
BATCH_SIZE = 2048
NUM_EPOCHS = 25
HIDDEN_SIZE = 128
N_LAYERS = 3
DROPOUT = 0.3
EMBEDDING_SIZE = 32

# Train parameters
TARGET = "EXPORT_centered"
FEATURES = [
  "contig", "comlang_off", "colony", "smctry",  # dist cepii categorical
]
N_SPLITS = 5
PATIENCE = 5
LEARNING_RATE = 0.01
WEIGHT_DECAY = 0.01
RANDOM_SEED = 16
KEEP_FRAC = 1.0
N_LAGS = 5
SUBSAMPLE_ENABLED = False
N_DYADS = 1000

SANCTION_COLS = ["arms", "military", "trade", "travel", "other", "financial"]

# Torch config
torch.manual_seed(RANDOM_SEED)
device = (
  torch.device("mps") if torch.backends.mps.is_available()
  else torch.device("cpu")
)

In [3]:
processed = pd.read_parquet(path="../../data/model/processed.parquet", engine="fastparquet")

df: DataFrame = processed.copy(deep=True)
df["dyad_id"] = df["ISO3_reporter"] + "_" + df["ISO3_partner"]
df = df.sort_values(by=["dyad_id", "Year"], ignore_index=True)

# Prepare sanction column as sum of all active boolean sanctions
df["sanction"] = (df[SANCTION_COLS]
                  .sum(axis=1)).astype(int)

# Coerce numerical columns to float
num_cols = ["distw", "GDP_reporter", "GDP_partner", "sanction", "contig",
            "comlang_off", "colony", "smctry", "Year", ]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").astype(float)

# Drop NA in numerical columns
df = df.dropna(subset=num_cols)

# Cast "Year" to integer
df["Year"] = df["Year"].astype(int)

# Cast "dyad_id" to categorical
for col in ["dyad_id"]:
  df[col] = pd.Categorical(df[col], categories=sorted(df[col].unique()))

# Define columns to be lagged and lag them while appending the number of the lag to the column name
lag_cols = ["GDP_reporter", "GDP_partner", "sanction"]
for col in lag_cols:
  for index in range(1, N_LAGS + 1):
    df[f"{col}_lag{index}"] = df.groupby("dyad_id", observed=True)[col].shift(index)

# Drop NA again for the lags that produced NA
df = df.dropna()

# Add lagged column names to the feature list
FEATURES += [f"{c}_lag{index}" for c in lag_cols for index in range(1, N_LAGS + 1)]

# Check time series for stationarity and cointegration

In [4]:
# Define which dyad to investigate
dyad_id = "USA_CHN"
dyad_df = df[df["dyad_id"] == dyad_id].sort_values("Year").copy()
dyad_df = dyad_df.set_index(pd.PeriodIndex(dyad_df["Year"], freq="Y"))

# Define which time series to investigate
ts_columns = [
  "GDP_reporter",
  "GDP_partner",
  "sanction",
  "EXPORT"
]

In [5]:
# Check for stationarity using Augmented Dickey-Fuller test
for col in ts_columns:

  try:
    result_adf = adfuller(dyad_df[col].values, autolag="AIC")
  except ValueError as e:
    print("!" * 50)
    print(f"⚠️ Column \"{col}\" constant over the whole dyad! Skipping!!")
    print(f"!" * 50 + "\n\n")
    continue

  adf_statistic = result_adf[0]
  p_value = result_adf[1]
  critical_values = result_adf[4]

  print(f"ADF Test for time series: {col}")
  print("=" * 50)

  print(f"p-value: {p_value}")
  print(f"ADF statistic: {adf_statistic}")
  print(
    f"Critical value 1%: {critical_values["1%"]}\nCritical value 5%: {critical_values["5%"]}\nCritical value 10%: {critical_values["10%"]}\n\n")

ADF Test for time series: GDP_reporter
p-value: 1.0
ADF statistic: 2.8316449590759802
Critical value 1%: -3.6790595944893187
Critical value 5%: -2.9678817237279103
Critical value 10%: -2.6231583472057074


ADF Test for time series: GDP_partner
p-value: 1.0
ADF statistic: 3.5172624062716515
Critical value 1%: -3.8092091249999998
Critical value 5%: -3.0216450000000004
Critical value 10%: -2.6507125


ADF Test for time series: sanction
p-value: 0.9573855641139912
ADF statistic: -0.013685510479879199
Critical value 1%: -3.6790595944893187
Critical value 5%: -2.9678817237279103
Critical value 10%: -2.6231583472057074


ADF Test for time series: EXPORT
p-value: 0.9737031806179908
ADF statistic: 0.2263842213865458
Critical value 1%: -3.6996079738860943
Critical value 5%: -2.9764303469999494
Critical value 10%: -2.627601001371742




In [6]:
# Check for stationarity using Kwiatkowski-Phillips-Schmidt-Shin test
for col in ts_columns:

  try:
    result_adf = kpss(dyad_df[col].values, regression="ct")
  except ValueError as error:
    print("!" * 50)
    print(f"⚠️ Column \"{col}\" constant over the whole dyad! Skipping!!")
    print(f"!" * 50 + "\n\n")
    continue

  kpss_statistic = result_adf[0]
  p_value = result_adf[1]
  critical_values = result_adf[3]

  print(f"KPSS Test for time series: {col}")
  print("=" * 50)

  print(f"p-value: {p_value}")
  print(f"KPSS statistic: {kpss_statistic}")
  print(
    f"Critical value 1%: {critical_values["1%"]}\nCritical value 2.5%: {critical_values["2.5%"]}\nCritical value 5%: {critical_values["5%"]}\nCritical value 10%: {critical_values["10%"]}\n\n")

KPSS Test for time series: GDP_reporter
p-value: 0.015859985718483054
KPSS statistic: 0.20037337141737852
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119


KPSS Test for time series: GDP_partner
p-value: 0.01
KPSS statistic: 0.21936577229818938
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119


KPSS Test for time series: sanction
p-value: 0.02089859481629792
KPSS statistic: 0.1869370804898722
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119


KPSS Test for time series: EXPORT
p-value: 0.1
KPSS statistic: 0.10361513501201357
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119




/var/folders/wz/kf7643gn3_s2867t_nnc9zxc0000gn/T/ipykernel_98134/612050169.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result_adf = kpss(dyad_df[col].values, regression="ct")
/var/folders/wz/kf7643gn3_s2867t_nnc9zxc0000gn/T/ipykernel_98134/612050169.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result_adf = kpss(dyad_df[col].values, regression="ct")


In [7]:
columns_coint = ["EXPORT", "GDP_reporter_lag2"]
columns_exog = ["sanction_lag1"]

In [8]:
# Using VAR to calculate the optimal amount of lags for< Johansen Test and VECM
var_data = dyad_df[columns_coint].dropna()
var_model = VAR(var_data)
var_selection = var_model.select_order(5)
var_result = var_model.fit(var_selection.aic)
k_ar_diff = max(var_result.k_ar - 1, 1)
k_ar_diff

4

In [9]:
# Check for cointegration using Johansen test
data_johansen = dyad_df[columns_coint].dropna()

jres = coint_johansen(data_johansen, det_order=1, k_ar_diff=k_ar_diff)
trace_statistic = jres.lr1[0]
max_eigenvalue_statistic = jres.lr2[0]
print(f"Johansen Test for cointegration between {columns_coint[0]} and {columns_coint[1]}")
print("=" * 50)
print("Trace Statistics:", jres.lr1)
print("Critical Values (Trace):", jres.cvt)

trace_stats = jres.lr1
crit_vals = jres.cvt

# Pick confidence level column (0=90%, 1=95%, 2=99%)
alpha_col = 1  # for 99% significance
rank = sum(trace_stats > crit_vals[:, alpha_col])
print(f"Estimated cointegration rank at 99%: {rank}")

Johansen Test for cointegration between EXPORT and GDP_reporter_lag2
Trace Statistics: [25.74362715  6.43443022]
Critical Values (Trace): [[16.1619 18.3985 23.1485]
 [ 2.7055  3.8415  6.6349]]
Estimated cointegration rank at 99%: 2


# VECM (Vector Error Correction Models)

Because GDP and EXPORT are both non-stationary and cointegrated, we cannot run the normal Granger causality test. But, we can run VECM.

In [12]:
Y = dyad_df[columns_coint].dropna()
X_exog = dyad_df[columns_exog].reindex(Y.index).fillna(0).astype(float)
vecm = VECM(endog=Y, exog=X_exog, k_ar_diff=k_ar_diff, coint_rank=rank, deterministic="ci")
vecm_result = vecm.fit()

# Save VECM summary to text file
summary_text = vecm_result.summary().as_text()

with open("../graph_output/vecm_summary.txt", "w") as f:
  f.write(summary_text)

vecm_result.summary()

,coef,std err,z,P>|z|,[0.025,0.975]
exog1,-2.884e+06,1.28e+06,-2.256,0.024,-5.39e+06,-3.79e+05
L1.EXPORT,0.3945,0.129,3.056,0.002,0.141,0.648
L1.GDP_reporter_lag2,1.357e-06,6.02e-06,0.225,0.822,-1.04e-05,1.32e-05
L2.EXPORT,0.0228,0.146,0.156,0.876,-0.264,0.309
L2.GDP_reporter_lag2,-2.31e-05,5.8e-06,-3.983,0.000,-3.45e-05,-1.17e-05
L3.EXPORT,0.0524,0.190,0.275,0.783,-0.321,0.426
L3.GDP_reporter_lag2,6.702e-08,5.11e-06,0.013,0.990,-9.95e-06,1.01e-05
L4.EXPORT,0.7416,0.177,4.191,0.000,0.395,1.088
L4.GDP_reporter_lag2,7.335e-06,5.63e-06,1.304,0.192,-3.69e-06,1.84e-05
,coef,std err,z,P>|z|,[0.025,0.975]
